# Real Out-of-Sample Validation for the Other Selectable Momentum Strategies

Epic 17: extends Epic 15/16's real walk-forward + pre-registered holdout methodology
(`notebooks/research/out_of_sample_validation.ipynb` / `out_of_sample_validation_weekly.ipynb`)
to the other selectable `strategy_type` values documented in `docs/MOMENTUM_STRATEGIES.md`.
Monthly regime only (`portfolio1`'s config as the base), weekly-regime-per-strategy-type
coverage remains a separate, un-closed gap, noted in `README.md`.

**Scope, confirmed by reading `core/strategy_signals.py`'s `resolve_strategy_scores()` directly,
not guessed**: of the 11 selectable values, `momentum` (Epic 15) and `relative_momentum`
("Explicit alias for `momentum`, byte-identical behavior") and `volatility_scaled_momentum`
(preset is `sizing_method: inverse_vol`, already `portfolio1`'s own default) need no separate
run. `hybrid_multi_factor` cannot be backtested at all (`generate_strategy_monthly_picks()`
raises `NotImplementedError`, no point-in-time fundamentals source exists). That leaves 6 real
variants for a full walk-forward + holdout run (`dual_momentum`, `correlation_weighted_momentum`,
`rank_sign_momentum`, `absolute_momentum`, `residual_momentum`, `path_dependent_momentum`) plus
`multi_timeframe_composite`, which needs different handling: its own scoring function never reads
the `lookback_period` argument at all (it uses `cfg.multi_timeframe_lookbacks` instead), so a
`lookback_period` grid search would be a meaningless no-op for it; it gets a single pre-registered
holdout evaluation instead, no walk-forward search.

**A real methodological bug found and fixed while building this, not a code bug, a test-harness
one**: building each variant's `BacktestConfig` from `dataclasses.asdict(portfolio1_cfg)`
(every field materialized, including ones at their default value) defeats
`daily_runner.apply_strategy_type_preset()`'s "only fill in fields the user hasn't already set"
contract, since a materialized dict can't tell "explicitly set" apart from "happens to equal the
default". Confirmed directly against a real run: `dual_momentum`/`correlation_weighted_momentum`/
`rank_sign_momentum` all produced BYTE-IDENTICAL results to plain `momentum` on the first attempt,
because `portfolio1`'s own `default_risk` already explicitly pins every field these 3 presets
would otherwise set (`sizing_method: inverse_vol`, `use_correlation_penalty: false`,
`use_absolute_momentum: false`). This is real, confirmed live-config behavior too, not just a
test artifact: selecting one of these 3 `strategy_type` values under a portfolio that relies on
the shipped `default_risk` block has ZERO effect vs. plain `momentum` unless the portfolio's own
`risk_overrides` ALSO explicitly re-overrides the differing field (`portfolio2` does this
correctly for `use_correlation_penalty`, confirmed in `config.yaml`). Fixed here by setting each
preset's own field value directly (matching `daily_runner.STRATEGY_TYPE_PRESETS` exactly), not by
changing any production code.

**A second real, expected (not a bug) finding**: `dual_momentum`'s backtest results below are
STILL byte-identical to plain `momentum`, even after the fix. This is correct, not broken:
`use_absolute_momentum` is documented LIVE-ONLY (`docs/MOMENTUM_STRATEGIES.md`, "no effect in the
backtest engine"), and `use_regime_filter` was already `true` in `portfolio1`'s own config, so
there is genuinely nothing left for this preset to change at the backtest level.

Same cached `crash_test_daily_prices.pkl` proxy panel as Epic 13/14/15/16, no new fetch. `SHY`
(already in the panel) substitutes for the default `defensive_ticker` (`"BIL"`, not in the proxy
universe) for `dual_momentum`/`absolute_momentum`, documented the same way as every prior epic's
own proxy-universe substitutions.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
import dataclasses
from dataclasses import replace

import pandas as pd

from momentum_trading.daily_runner import load_config, apply_strategy_type_preset
from momentum_trading.core import functions_quant_extensions as fnx
from momentum_trading.core.strategy_signals import generate_strategy_monthly_picks
from momentum_trading.backtest.momentum_backtest import BacktestConfig, run_custom_backtest

## 1. Load the real config and the cached proxy-universe price history

In [ ]:
config = load_config()
portfolio1_cfg = config["portfolios_resolved"]["portfolio1"]["cfg"]

PROXY_TICKERS = [
    "SPY", "QQQ", "DIA", "XLK", "XLF", "XLE", "XLI", "XLP", "XLU", "XLV", "XLY",
    "GLD", "TLT", "IEF", "SHY", "LQD", "IWM",
]

daily_prices = pd.read_pickle("crash_test_daily_prices.pkl")
print(daily_prices.shape, daily_prices.index.min(), "->", daily_prices.index.max())

## 2. Pre-registered train / holdout split, and a config builder

In [ ]:
train, holdout = fnx.pre_registered_split(daily_prices, split_date="2015-01-01")
print(f"Train:   {train.index.min().date()} to {train.index.max().date()} ({len(train)} rows)")
print(f"Holdout: {holdout.index.min().date()} to {holdout.index.max().date()} ({len(holdout)} rows)")

LOOKBACK_CANDIDATES = [6, 9, 12, 15, 18]


def build_cfg(strategy_type, overrides=None):
    """Reuses apply_strategy_type_preset() the same way daily_runner.load_config() does, but
    each preset-driven variant\'s field is set EXPLICITLY here too (see the note above on why
    starting from a fully-materialized dataclass dict alone silently defeats the preset when
    portfolio1\'s own default_risk already pins that same field)."""
    merged = dataclasses.asdict(portfolio1_cfg)
    merged["strategy_type"] = strategy_type
    if overrides:
        merged.update(overrides)
    merged = apply_strategy_type_preset(merged)
    return BacktestConfig(**merged)


# Matches daily_runner.STRATEGY_TYPE_PRESETS exactly, plus the SHY defensive_ticker
# substitution (BIL is not in the cached proxy universe).
FULL_SEARCH_VARIANTS = {
    "dual_momentum": {"use_absolute_momentum": True, "use_regime_filter": True, "defensive_ticker": "SHY"},
    "correlation_weighted_momentum": {"use_correlation_penalty": True},
    "rank_sign_momentum": {"sizing_method": "equal_weight"},
    "absolute_momentum": {"defensive_ticker": "SHY"},
    "residual_momentum": {},
    "path_dependent_momentum": {},
}

## 3. Walk-forward + holdout + bootstrap CI, for each of the 6 full-search variants

Same real pipeline as Epic 15/16, one loop over the 6 variants, collecting one combined summary
row each.

In [ ]:
summary_rows = []

for strategy_type, overrides in FULL_SEARCH_VARIANTS.items():
    cfg = build_cfg(strategy_type, overrides)

    wf_results = fnx.run_walk_forward_lookback_search(
        train, PROXY_TICKERS, cfg,
        lookback_candidates=LOOKBACK_CANDIDATES,
        train_years=4, test_years=1, step_years=1,
        metric="Sharpe",
    )
    if wf_results.empty:
        summary_rows.append({"strategy_type": strategy_type, "n_folds": 0})
        continue

    chosen_lookback = int(wf_results["chosen_lookback"].mode().iloc[0])
    mean_train_sharpe = wf_results["train_Sharpe"].mean()
    mean_test_sharpe = wf_results["test_Sharpe"].mean()

    holdout_cfg = replace(cfg, lookback_period=chosen_lookback)
    holdout_start = holdout.index.min()
    picks_full = generate_strategy_monthly_picks(
        daily_prices, PROXY_TICKERS, holdout_cfg, chosen_lookback, holdout_cfg.top_n,
    )
    picks_holdout = picks_full[picks_full.index >= holdout_start]
    holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **holdout_cfg.__dict__)
    ts = holdout_bt.attrs.get("tearsheet", {})

    holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()
    ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)

    summary_rows.append({
        "strategy_type": strategy_type,
        "n_folds": len(wf_results),
        "chosen_lookback": chosen_lookback,
        "mean_train_sharpe": round(mean_train_sharpe, 2),
        "mean_test_sharpe": round(mean_test_sharpe, 2),
        "holdout_sharpe": round(ts.get("Sharpe", float("nan")), 2),
        "holdout_cagr": round(ts.get("CAGR", float("nan")) * 100, 2),
        "holdout_alpha": round(ts.get("Alpha", float("nan")) * 100, 2),
        "ci_low": round(ci["ci_low"], 2),
        "ci_high": round(ci["ci_high"], 2),
    })
    print(f"{strategy_type}: done")

## 4. `multi_timeframe_composite`: holdout-only, no lookback grid search

Its own scoring never reads `lookback_period` (uses `cfg.multi_timeframe_lookbacks` instead), so
a grid search over it would be a no-op. A single pre-registered holdout evaluation instead,
`portfolio1`'s configured `multi_timeframe_lookbacks` default (`[3, 6, 12]`).

In [ ]:
mtc_cfg = build_cfg("multi_timeframe_composite")
holdout_start = holdout.index.min()
picks_full = generate_strategy_monthly_picks(
    daily_prices, PROXY_TICKERS, mtc_cfg, mtc_cfg.lookback_period, mtc_cfg.top_n,
)
picks_holdout = picks_full[picks_full.index >= holdout_start]
holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **mtc_cfg.__dict__)
ts = holdout_bt.attrs.get("tearsheet", {})

holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()
ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)

summary_rows.append({
    "strategy_type": "multi_timeframe_composite (holdout-only)",
    "n_folds": None,
    "chosen_lookback": None,
    "mean_train_sharpe": None,
    "mean_test_sharpe": None,
    "holdout_sharpe": round(ts.get("Sharpe", float("nan")), 2),
    "holdout_cagr": round(ts.get("CAGR", float("nan")) * 100, 2),
    "holdout_alpha": round(ts.get("Alpha", float("nan")) * 100, 2),
    "ci_low": round(ci["ci_low"], 2),
    "ci_high": round(ci["ci_high"], 2),
})

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))